In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# 通用 MultiHeadAttention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, attn_mask=None):
        batch_size, tgt_len, _ = query.size()
        _, src_len, _ = key.size()
        
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)
        
        Q = Q.view(batch_size, tgt_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, src_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, src_len, self.num_heads, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if attn_mask is not None:
            scores = scores + attn_mask
        
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        out = torch.matmul(weights, V)
        
        out = out.transpose(1, 2).contiguous().view(batch_size, tgt_len, self.d_model)
        out = self.W_O(out)
        return out, weights

# ------------------------encoder--------------------------#
class PositionwiseFeedForwar1d(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))
    
    
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x
    
    
class TransformerEncoderLayer(nn.Module):
    def __init__(self,d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        attn_output, _ = self.self_attn(x, x, x)
        x = self.norm1(x + self.dropout(attn_output))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x
    

class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, num_heads=8, d_ff=2048, num_layers=6,max_len=5000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)
    
#---------------------------decoder-----------------------------#
def generate_subsequent_mask(seq_len, device=None):
    mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
    mask = mask.masked_fill(mask == 1, float('-inf')).masked_fill(mask == 0, float(0.0))
    return mask

# 可选
def create_padding_mask(pad_mask):
    if pad_mask is None:
        return None
    additive = pad_mask.unsqueeze(1).unsqueeze(1).to(torch.float32) * float('-inf')
    return additive
    
# FFN
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        return self.fc2(self.dropout(F.relu(self.fc1(x))))
    
# decoder layer
class TransformerDecoderLayer(nn.Module): 
    def __init__(self, d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        """_summary_

        Args:
            tgt (_type_): (batch, tgt_len, d_model)  -- decoder input embeddings
            memory (_type_): (batch, src_len, d_model)  -- encoder outputs
            tgt_mask (_type_, optional): _description_. Defaults to None.
            memory_mask (_type_, optional): _description_. Defaults to None.
        """
        _tgt, self_w = self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask)
        tgt = self.norm1(tgt + self.dropout(_tgt))
        
        _tgt2, cross_w = self.cross_attn(tgt, memory, memory, attn_mask=memory_mask)
        tgt = self.norm2(tgt + self.dropout(_tgt2))
        
        _tgt3 = self.ffn(tgt)
        tgt = self.norm3(tgt + self.dropout(_tgt3))
        
        return tgt, self_w, cross_w
    
# decoder
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, num_layers=6, d_model=512, num_heads=8, d_ff=2048, dropout=0.1, max_len=5000):
        super().__init__()
        # 要加embedding，否则tgt少一个维度
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        x = self.embedding(tgt) * math.sqrt(self.embedding.embedding_dim)
        x = self.pos_encoding(x)
        
        attn_weights_self =[]
        attn_weights_cross = []
        
        for layer in self.layers:
            x, w_self, w_cross = layer(x, memory, tgt_mask=tgt_mask, memory_mask=memory_mask)
            attn_weights_self.append(w_self)
            attn_weights_cross.append(w_cross)
        x = self.norm(x)
        return x, attn_weights_self, attn_weights_cross
    
# -------------------------合成transformer------------------------#
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8, num_encoder_layers=6, num_decoder_layers=6, d_ff=2048, dropout=0.1, max_len=5000):
        super().__init__()
        self.encoder = TransformerEncoder(src_vocab_size, d_model, num_heads, d_ff, num_encoder_layers, max_len)
        self.decoder = TransformerDecoder(tgt_vocab_size, num_decoder_layers, d_model, num_heads, d_ff, dropout, max_len)
        self.output_linear = nn.Linear(d_model, tgt_vocab_size)
        
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        memory = self.encoder(src)
        out, attn_self, attn_cross = self.decoder(tgt, memory, tgt_mask=tgt_mask, memory_mask=None)
        logits = self.output_linear(out)
        
        return logits, attn_self, attn_cross
    
# ------------------------测试----------------------------#
src_vocab_size = 1000
tgt_vocab_size = 1000
batch_size = 2
src_len = 7
tgt_len = 5

model = Transformer(src_vocab_size, tgt_vocab_size, d_model=32, num_heads=4, num_encoder_layers=2, num_decoder_layers=2, d_ff=64)

# 准备mask
src = torch.randint(0, src_vocab_size, (batch_size, src_len))
tgt = torch.randint(0, tgt_vocab_size, (batch_size, tgt_len))

tgt_mask = generate_subsequent_mask(tgt_len).unsqueeze(0).unsqueeze(0)

logits, attn_self, attn_cross = model(src, tgt, tgt_mask=tgt_mask)

print("logits.shape: ", logits.shape)

logits.shape:  torch.Size([2, 5, 1000])


## 英文 -> 中文 翻译任务

In [2]:
# 数据集准备

# mini test
src_texts = ["I love you", "How are you", "Thank you", "Good morning"]
tgt_texts = ["我爱你", "你好吗", "谢谢你", "早上好"]

# from datasets import load_dataset

# dataset = load_dataset("tatoeba", lang1='en', lang2='zh')
# train_data = dataset['train']

# src_texts = [x['translation']['en'] for x in train_data]
# tgt_texts = [x['translation']['zh'] for x in train_data]

# import json

# with open("./data/iwslt2017-en-zh-train/iwslt2017-en-zh-train.json", "r", encoding="utf-8") as f:
#     data = json.load(f)  # 加载为Python列表

# # 提取所有英文和中文句子
# src_texts = [x["translation"]["en"] for x in data]
# tgt_texts = [x["translation"]["zh"] for x in data]

In [3]:
# 分词与词表
from collections import Counter

def build_vocab(sentences):
    vocab = {"<pad>": 0, "<bos>": 1, "<eos>": 2, "<unk>": 3}
    idx = 4
    for s in sentences:
        for ch in s:
            if ch not in vocab:
                vocab[ch] = idx
                idx += 1
    return vocab

def encode(text, vocab, max_len=20):
    tokens = [vocab.get(ch, vocab["<unk>"]) for ch in text]
    tokens = [vocab["<bos>"]] + tokens + [vocab["<eos>"]]
    tokens = tokens[:max_len] + [vocab["<pad>"]] * (max_len - len(tokens))
    return tokens

src_vocab = build_vocab(src_texts)
tgt_vocab = build_vocab(tgt_texts)
src_vocab_size = len(src_vocab)
tgt_vocab_size = len(tgt_vocab)

# 编码
src_ids = torch.tensor([encode(s, src_vocab) for s in src_texts])
tgt_ids = torch.tensor([encode(s, tgt_vocab) for s in tgt_texts])

In [4]:
# 定义模型并训练
model = Transformer(src_vocab_size=src_vocab_size, tgt_vocab_size=tgt_vocab_size, d_model=128, num_heads=4, num_encoder_layers=2, num_decoder_layers=2)

criterion = nn.CrossEntropyLoss(ignore_index=0) # 忽略pad?????????
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    
    tgt_input = tgt_ids[:, :-1]
    tgt_output = tgt_ids[:, 1:]
    
    tgt_mask = generate_subsequent_mask(tgt_input.size(1))
    
    logits,_,_ = model(src_ids, tgt_input, tgt_mask=tgt_mask)
    
    loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output.reshape(-1))
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}, loss={loss.item():.4f}")


Epoch 20, loss=0.3409
Epoch 40, loss=0.1315
Epoch 60, loss=0.0599
Epoch 80, loss=0.1959
Epoch 100, loss=0.0797
Epoch 120, loss=0.0230
Epoch 140, loss=0.0148
Epoch 160, loss=0.0252
Epoch 180, loss=0.0636
Epoch 200, loss=0.0301


## 推理

In [5]:
def translate(model, sentence, src_vocab, tgt_vocab, max_len=20):
    model.eval()
    inv_tgt_vocab = {v:k for k, v in tgt_vocab.items()}
    
    src = torch.tensor([encode(sentence, src_vocab)])
    memory = model.encoder(src)
    tgt = torch.tensor([[tgt_vocab["<bos>"]]])
    
    for _ in range(max_len):
        tgt_mask = generate_subsequent_mask(tgt.size(1))
        logits,_,_ = model(src, tgt, tgt_mask=tgt_mask)
        next_token = logits[:, -1, :].argmax(-1).unsqueeze(0)
        tgt = torch.cat([tgt, next_token], dim=1)
        
        if next_token.item() == tgt_vocab["<eos>"]:
            break
    
    return ''.join([inv_tgt_vocab[i.item()] for i in tgt[0][1:-1]])

test_sentence = "I love you"
print(translate(model, test_sentence, src_vocab, tgt_vocab))

我爱你
